In [1]:
import pandas as pd
from db_bootstrap import get_engine
from sqlalchemy import text

# Get database connection
engine = get_engine()

# Load data from price_history table
df_price_history = pd.read_sql_query(
    "SELECT * FROM price_history;", 
    engine
)

# print(f"✅ Loaded {len(df_price_history)} rows from price_history table")
# print(f"\nDataframe shape: {df_price_history.shape}")
# print(f"\nColumn names:\n{df_price_history.columns.tolist()}")
# print(f"\nData types:\n{df_price_history.dtypes}")


print(df_price_history.isnull().sum()[df_price_history.isnull().sum() > 0])

volume_30d_avg      30
volume_ratio        30
sma_20              19
sma_50              49
sma_200           5858
ema_20              19
rsi_14              14
macd                25
macd_signal         33
macd_hist           33
bb_upper            19
bb_lower            19
bb_mid              19
vs_nifty_pct        48
dtype: int64


In [2]:
# Check for null values
print("\n" + "="*60)
print("NULL VALUES ANALYSIS")
print("="*60)

null_counts = df_price_history.isnull().sum()
null_percentages = (df_price_history.isnull().sum() / len(df_price_history) * 100).round(2)

null_summary = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Null_Percentage': null_percentages.values
})

# Show only columns with nulls
null_summary_filtered = null_summary[null_summary['Null_Count'] > 0].sort_values('Null_Count', ascending=False)

if len(null_summary_filtered) == 0:
    print("✅ No null values found in price_history table!")
else:
    print(f"\n⚠️  Found nulls in {len(null_summary_filtered)} columns:\n")
    print(null_summary_filtered.to_string(index=False))
    
print(f"\n\nTotal null values in entire dataframe: {df_price_history.isnull().sum().sum()}")
print(f"Total cells in dataframe: {df_price_history.shape[0] * df_price_history.shape[1]}")

print()



NULL VALUES ANALYSIS

⚠️  Found nulls in 14 columns:

        Column  Null_Count  Null_Percentage
       sma_200        5858            59.03
        sma_50          49             0.49
  vs_nifty_pct          48             0.48
     macd_hist          33             0.33
   macd_signal          33             0.33
  volume_ratio          30             0.30
volume_30d_avg          30             0.30
          macd          25             0.25
        ema_20          19             0.19
        sma_20          19             0.19
      bb_lower          19             0.19
      bb_upper          19             0.19
        bb_mid          19             0.19
        rsi_14          14             0.14


Total null values in entire dataframe: 6215
Total cells in dataframe: 327459



In [3]:
# Load data from three Excel files
import os
import subprocess
import sys

# Ensure openpyxl is installed
try:
    import openpyxl
except ImportError:
    print("Installing openpyxl...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])
    import openpyxl

excel_files = {
    'raw_stock_data': 'raw_stock_data_excel.xlsx',
    'stock_data': 'stock_data_excel.xlsx',
    'price_history': 'price_history_excel.xlsx'
}

project_dir = r'c:\adani uni\sem_6\data_vis\project'
excel_data = {}

print("="*60)
print("LOADING EXCEL FILES")
print("="*60)

for table_name, filename in excel_files.items():
    filepath = os.path.join(project_dir, filename)
    
    try:
        # Read Excel file
        df = pd.read_excel(filepath)
        excel_data[table_name] = df
        
        print(f"\n✅ Loaded {filename}")
        print(f"   Shape: {df.shape} (rows, columns)")
        print(f"   Columns: {df.columns.tolist()}")
        
    except Exception as e:
        print(f"\n❌ Error loading {filename}: {str(e)}")

print("\n" + "="*60)
print("EXCEL FILES SUMMARY")
print("="*60)
for table_name, df in excel_data.items():
    print(f"\n{table_name}: {df.shape[0]} rows × {df.shape[1]} columns")


LOADING EXCEL FILES

✅ Loaded raw_stock_data_excel.xlsx
   Shape: (50, 20) (rows, columns)
   Columns: ['date', 'ticker', 'company_name', 'sector', 'industry', 'open', 'high', 'low', 'close', 'volume', 'volume_30d_avg', 'volume_ratio', 'market_cap_cr', 'pe_ratio', 'roe_pr', 'profit_margin_pr', 'debt_to_equity', 'week52_high', 'week52_low', 'qoq_revenue_growth_pr']

✅ Loaded stock_data_excel.xlsx
   Shape: (50, 20) (rows, columns)
   Columns: ['date', 'ticker', 'company_name', 'sector', 'industry', 'open', 'high', 'low', 'close', 'volume', 'volume_30d_avg', 'volume_ratio', 'market_cap_cr', 'pe_ratio', 'roe_pr', 'profit_margin_pr', 'debt_to_equity', 'week52_high', 'week52_low', 'qoq_revenue_growth_pr']

✅ Loaded price_history_excel.xlsx
   Shape: (9923, 37) (rows, columns)
   Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'date', 'ticker', 'company_name', 'sector', 'industry', 'open', 'high', 'low', 'close', 'volume', 'volume_30d_avg', 'volume_ratio', 'market_cap_cr', 

In [4]:
# Load Excel data into PostgreSQL (maintaining both sources)
print("\n" + "="*60)
print("LOADING EXCEL DATA INTO POSTGRESQL")
print("="*60)

for table_name, df in excel_data.items():
    try:
        # Use 'append' mode to add to existing data, 'replace' to overwrite
        # 'append' keeps both Excel and existing DB data
        df.to_sql(table_name, engine, if_exists='append', index=False)
        
        print(f"\n✅ Loaded {table_name} ({df.shape[0]} rows)")
        print(f"   Status: Data appended to PostgreSQL (maintaining both sources)")
        
    except Exception as e:
        print(f"\n❌ Error loading {table_name} to DB: {str(e)}")

print("\n" + "="*60)
print("VERIFYING DATA IN POSTGRESQL")
print("="*60)

# Verify data was loaded by checking row counts
for table_name in excel_data.keys():
    try:
        result = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table_name};", engine)
        count = result['count'].iloc[0]
        print(f"\n✅ {table_name}: {count} total rows in PostgreSQL")
        
    except Exception as e:
        print(f"\n⚠️  {table_name}: Table may not exist yet - {str(e)}")



LOADING EXCEL DATA INTO POSTGRESQL

✅ Loaded raw_stock_data (50 rows)
   Status: Data appended to PostgreSQL (maintaining both sources)

✅ Loaded stock_data (50 rows)
   Status: Data appended to PostgreSQL (maintaining both sources)

❌ Error loading price_history to DB: Execution failed on sql 'INSERT INTO price_history ("Unnamed: 0", "Unnamed: 1", "Unnamed: 2", "Unnamed: 3", date, ticker, company_name, sector, industry, open, high, low, close, volume, volume_30d_avg, volume_ratio, market_cap_cr, pe_ratio, roe_pr, profit_margin_pr, debt_to_equity, week52_high, week52_low, nifty_index_close, sma_20, sma_50, sma_200, ema_20, rsi_14, macd, macd_signal, macd_hist, bb_upper, bb_lower, bb_mid, vs_nifty_pct, vs_nifty_cumulative) VALUES (:UnnamedC_0, :UnnamedC_1, :UnnamedC_2, :UnnamedC_3, :date, :ticker, :company_name, :sector, :industry, :open, :high, :low, :close, :volume, :volume_30d_avg, :volume_ratio, :market_cap_cr, :pe_ratio, :roe_pr, :profit_margin_pr, :debt_to_equity, :week52_high, :

In [5]:
# Display sample data from each table
print("\n" + "="*60)
print("SAMPLE DATA FROM EACH TABLE")
print("="*60)

for table_name in excel_data.keys():
    print(f"\n📊 {table_name.upper()} - First 5 rows:")
    print("-" * 60)
    try:
        df_sample = pd.read_sql_query(
            f"SELECT * FROM {table_name} LIMIT 5;", 
            engine
        )
        print(df_sample.to_string(index=False))
    except Exception as e:
        print(f"Could not retrieve data: {str(e)}")

print("\n" + "="*60)
print("✅ LOAD COMPLETE - Both Excel and PostgreSQL are synchronized")
print("="*60)



SAMPLE DATA FROM EACH TABLE

📊 RAW_STOCK_DATA - First 5 rows:
------------------------------------------------------------
      date     ticker                                  company_name             sector                industry   open   high    low  close  volume  volume_30d_avg  volume_ratio  market_cap_cr  pe_ratio  roe_pr  profit_margin_pr  debt_to_equity  week52_high  week52_low  qoq_revenue_growth_pr
2026-04-19   ADANIENT                     Adani Enterprises Limited             Energy            Thermal Coal 2240.0 2240.0 2186.1 2218.3 2408901       2259094.0          1.07      305011.75 20.092150   11.18             14.11         177.767       2695.0      1753.0                    NaN
2026-04-19 ADANIPORTS Adani Ports and Special Economic Zone Limited        Industrials         Marine Shipping 1559.0 1578.5 1549.2 1573.4 6024587       3329569.0          1.81      362274.51 27.232420   19.41             34.24          81.556       1584.0      1181.2                   6.34
